<h1 align="center">Laboratorio 8</h1>

## Información

**Integrantes:**

| Name              | Institution ID | GitHub User |
| ----------------- | -------------- | ----------- |
| Josué Say         | 22801          | JosueSay    |
| Carlos Valladares | 221164         | vgcarlol    |

- [Repositorio](https://github.com/JosueSay/intro-to-computer-vision/tree/main/labs/lab8)

## Preparación de entorno

In [ ]:
# %pip install -r requirements.txt
# jupyter nbconvert lab8.ipynb --to html

**DataSet:** <https://github.com/eg4000/SKU110K_CVPR19>

Usted forma parte del equipo de Inteligencia Artificial de VisorShelf, una startup guatemalteca de tecnología retail que desarrolla un sistema de auditoría automática de anaqueles para supermercados y tiendas de conveniencia. El sistema debe identificar y localizar productos en imágenes tomadas por cámaras fijas instaladas frente a los anaqueles, con el objetivo de detectar quiebres de stock, productos mal ubicados y desorden en la exhibición.

La restricción operativa principal: las cámaras capturan imágenes cada 30 segundos y el sistema debe procesar cada imagen en menos de $500 , ms$ corriendo on-premise en hardware de tienda (CPU de gama media, sin GPU dedicada). Su equipo dispone de un dataset anotado con bounding boxes de productos en anaquel.


## Task 1

El siguiente conjunto de preguntas evalúa su capacidad de analizar, justificar y conectar los fundamentos matemáticos de la detección de objetos con decisiones de ingeniería reales dentro del proyecto VisorShelf. No basta con enunciar fórmulas: se espera que usted explique el significado de cada término y argumente su relevancia práctica en el contexto de auditoría de anaqueles.

### Pregunta 1.1

El gerente de producto de VisorShelf le presenta la siguiente situación: el sistema detecta una lata de atún en el anaquel y devuelve la caja predicha
$b' = (142, 89, 218, 165)$ en formato $(x_{min}, y_{min}, x_{max}, y_{max})$.

El radiólogo de calidad del cliente anota manualmente la caja real
$b^* = (138, 84, 222, 170)$.

El cliente pregunta: “¿Qué tan buena es esa detección?”

Con esto en mente, responda las siguientes preguntas en su reporte:

1. Calcule manualmente el IoU entre las dos cajas. Muestre paso a paso el cálculo del área de intersección, el área de unión y el valor final. Explique en términos no técnicos qué significa ese número para el cliente de VisorShelf.

2. En la fórmula
   $IoU = \frac{|I|}{|U|}$,
   identifique qué representa cada símbolo ($|I|$, $|U|$) y explique por qué el denominador es la unión y no el área del ground truth. ¿Qué problema concreto evita esa decisión de diseño?

3. El equipo de VisorShelf está evaluando dos umbrales de IoU para decidir si una detección es válida:
   $\theta = 0.5$ y $\theta = 0.75$.
   ¿Cuál recomendaría para el sistema de auditoría de anaqueles y por qué? Considere el impacto operativo de los falsos positivos y falsos negativos en el negocio del cliente.

### Pregunta 1.2

Durante una prueba piloto en una tienda de conveniencia, el detector de VisorShelf analiza una imagen con 15 productos en el anaquel. El modelo genera 18 predicciones. Tras aplicar el umbral $IoU = 0.5$, el equipo clasifica:
12 TP, 6 FP y 3 FN.

Con base a esto, responda dentro de su reporte:

4. Calcule la Precisión y el Recall para esta prueba. En las fórmulas
   $P = \frac{TP}{TP + FP}$
   y
   $R = \frac{TP}{TP + FN}$,
   explique verbalmente qué mide cada término del denominador y por qué ambas métricas son necesarias para evaluar el sistema.

5. El director de operaciones de la tienda le dice: “Prefiero que el sistema no se pierda ningún quiebre de stock, aunque a veces nos avise de falsas alarmas.” Traduzca esa preferencia a términos de Precisión y Recall. ¿Qué umbral de confianza ajustaría y en qué dirección?

6. Explique qué es el mAP (Mean Average Precision) y por qué es más informativo que reportar un único valor de Precisión o Recall. En su explicación, distinga entre
   $mAP@0.5$ (protocolo PASCAL VOC) y
   $mAP@0.5:0.95$ (protocolo COCO),
   y argumente cuál protocolo sería más exigente para VisorShelf y por qué.

### Pregunta 1.3

En una imagen de anaquel con 40 productos, el modelo de VisorShelf genera 312 cajas candidatas antes de cualquier postprocesamiento. El cliente observa el resultado intermedio y exclama: “¡El sistema está viendo el mismo producto decenas de veces!”

Responda lo siguiente en su reporte:

7. Explique al cliente qué es el Non-Maximum Suppression (NMS) y por qué el detector genera múltiples cajas para el mismo objeto. Describa el algoritmo paso a paso en lenguaje no técnico.

8. El parámetro $\theta_{NMS}$ controla qué tan agresivo es el NMS al suprimir cajas. En un anaquel densamente poblado donde los productos están muy juntos, ¿qué valor de $\theta_{NMS}$ recomendaría (alto o bajo) y por qué? Argumente el riesgo en cada dirección.

9. ¿En qué orden se deben aplicar el umbral de confianza y el NMS? Justifique la respuesta y explique qué sucedería computacionalmente si se invierte ese orden en un sistema que procesa 30 imágenes por minuto.

## Task 2

Las siguientes preguntas evalúan su comprensión estratégica de la evolución de los detectores de dos etapas y su capacidad de tomar decisiones de arquitectura justificadas dentro del contexto operativo de VisorShelf. Se valorará la coherencia del argumento con las restricciones reales del sistema.

### Pregunta 2.1

El CTO de VisorShelf propone usar el detector original R-CNN (2014) para la primera versión del sistema. El equipo de ingeniería calcula que con el dataset actual y una CPU de tienda, cada imagen tardaría aproximadamente 45 segundos en procesarse.

Con esto responda en su reporte:

1. Identifique el cuello de botella principal de R-CNN que causa esa latencia. Explique por qué procesar 2,000 propuestas de región de forma independiente es computacionalmente costoso, conectando su respuesta con lo que la red hace internamente en cada pasada.

2. Fast R-CNN introdujo el feature map compartido y el RoI Pooling para resolver ese cuello de botella. Explique la lógica detrás de cada uno, ¿qué cómputo elimina el feature map compartido?, ¿qué problema resuelve el RoI Pooling y qué operación matemática realiza para producir un tensor de tamaño fijo a partir de regiones de tamaño variable?

3. Con Fast R-CNN el tiempo de CNN bajó a $0.3 , s$/imagen, pero el tiempo total seguía siendo $\sim 2.3 , s$. Identifique el nuevo cuello de botella y explique por qué Selective Search representa un problema arquitectónico más profundo que simplemente ser lento.

### Pregunta 2.2

El equipo de VisorShelf decide usar Faster R-CNN como base del sistema. Un ingeniero junior pregunta:
“¿Por qué necesitamos una Region Proposal Network si ya tenemos el feature map? ¿No podríamos simplemente hacer sliding window directamente sobre el feature map?”

Responda en su reporte:

4. Responda la pregunta del ingeniero junior. Explique qué hace la RPN que un sliding window clásico no puede hacer, y por qué el hecho de que la RPN opere sobre el mismo feature map del backbone es una ventaja semántica, no solo computacional.

5. La RPN utiliza anchor boxes predefinidos (9 por posición: 3 escalas × 3 relaciones de aspecto). Explique qué son los anchors y por qué la red predice deltas $(\Delta x, \Delta y, \Delta w, \Delta h)$ en lugar de coordenadas absolutas. En la decodificación
   $w = w_a \cdot e^{\Delta w}$,
   identifique qué representa cada símbolo y explique por qué se usa la exponencial para las dimensiones.

6. Faster R-CNN logra $\sim 5 , FPS$ con VGG16. La restricción de VisorShelf es procesar una imagen en menos de $500 , ms$ ($\geq 2 , FPS$). Tomando en cuenta esa restricción, ¿recomendaría Faster R-CNN para producción en el hardware de tienda? Argumente su respuesta considerando al menos dos factores distintos a la velocidad pura (p. ej., precisión en objetos pequeños y densos, facilidad de fine-tuning, ecosistema de implementación).

### Pregunta 2.3

El equipo de producto presenta dos propuestas para el detector de producción de VisorShelf:

| Dimensión                           | Propuesta A: Faster R-CNN + ResNet-50 + FPN | Propuesta B: YOLOv8n (nano) fine-tuned  |
| ----------------------------------- | ------------------------------------------- | --------------------------------------- |
| Velocidad estimada                  | 8–12 FPS en GPU / $\sim 1.5$ FPS en CPU     | 60–80 FPS en GPU / $\sim 15$ FPS en CPU |
| mAP@0.5 (referencia)                | $\sim 55$ en COCO                           | $\sim 37$ en COCO                       |
| Tamaño del modelo                   | $\sim 160 , MB$                             | $\sim 6 , MB$                           |
| Manejo de objetos pequeños y densos | Alto (FPN multi-escala)                     | Moderado                                |
| Costo de fine-tuning                | Moderado (pipeline de dos etapas)           | Bajo (arquitectura simple)              |

Responda en su reporte:

7. Tomando en cuenta las restricciones operativas de VisorShelf (hardware sin GPU, latencia < $500 , ms$, anaqueles densos), ¿cuál propuesta recomendaría y por qué? No se limite a comparar los números de la tabla; argumente la lógica de la decisión conectándola con las características técnicas de cada arquitectura.

8. Si VisorShelf logra instalar una GPU de gama media (RTX 3060) en las tiendas flagship, ¿cambiaría su recomendación? Explique cómo ese cambio de hardware altera el trade-off entre las dos propuestas.

9. ¿Qué riesgo técnico específico introduce hacer fine-tuning de Faster R-CNN con un dataset de anaqueles sin aplicar learning rate diferenciado entre el backbone y las capas nuevas? ¿Cómo se llama ese fenómeno y cómo lo mitigaría?

## Task 3

Utilice PyTorch o TensorFlow/Keras a su elección. No se proporciona código base; usted debe construir su solución apoyándose en la documentación oficial, recursos académicos y su criterio de ingeniería. Ejecute sus experimentos en Google Colab, Kaggle Notebooks o GPU local. Entregue el enlace al notebook con todas las celdas ejecutadas y los resultados visibles. El notebook debe estar limpio, comentado y reproducible.

La evaluación considera no solo que el código funcione, sino que usted entienda cada decisión que tomó y la justifique en su reporte.

Con esto realice lo siguiente:

### 1. Preparación del Dataset:

a. **Dataset:** Descargue un subconjunto del dataset SKU110K (mínimo 500 imágenes de entrenamiento, 100 de validación, 100 de prueba). Si el dataset completo no es accesible, puede utilizar Grocery Store Dataset o Open Images V7 filtrado por categorías de productos de anaquel. Documente en su reporte la fuente exacta y el proceso de descarga.

b. **Preprocesamiento:** Asegúrese de que las anotaciones estén en formato compatible con el detector elegido (COCO JSON, YOLO `.txt`, o Pascal VOC XML). Justifique en su reporte cualquier conversión que realice y documente la distribución de clases del subconjunto utilizado.

c. **Verificación:** Visualice al menos 5 imágenes con sus bounding boxes anotados antes del entrenamiento. Incluya esas visualizaciones en su notebook.

### 2. Entrenamiento de Dos Detectores:

a. Seleccione **dos detectores pre-entrenados** de su elección (por ejemplo: Faster R-CNN con ResNet-50, SSD, DETR, RT-DETR, entre otros). Para cada uno:

i. Cargue el modelo pre-entrenado en COCO o ImageNet y adapte el cabezal de clasificación al número de clases del dataset de anaqueles. Justifique en su reporte por qué eligió esos dos modelos específicos para compararlos.

ii. Realice **fine-tuning** con las capas base inicialmente congeladas. Documente los hiperparámetros elegidos (learning rate, épocas, batch size, optimizador) y argumente por qué son razonables para este problema y tamaño de dataset.

iii. Implemente **Early Stopping** monitoreando la métrica de validación apropiada. Explique en su reporte qué métrica eligió y la lógica detrás de detener el entrenamiento anticipadamente.

iv. Guarde los pesos del mejor modelo de cada arquitectura.

### 3. Evaluación

a. Para cada modelo, calcule y registre las siguientes métricas sobre el conjunto de prueba:

i. $mAP@0.5$
ii. $mAP@0.5:0.95$
iii. Precisión y Recall para la clase de producto principal
iv. FPS promedio de inferencia sobre 50 imágenes
v. Tamaño del modelo guardado en disco (MB)

b. Visualice al menos 3 imágenes de prueba con las detecciones del mejor modelo superpuestas, mostrando las cajas predichas con su score de confianza.

## Dictamen Ejecutivo:

Redacte en su reporte un dictamen ejecutivo de 1 a 2 páginas dirigido al CTO de VisorShelf. El dictamen debe incluir:

* **Tabla comparativa** cruzando ambos modelos con todas las métricas del Paso 3.

* **Análisis de velocidad vs. precisión:** ¿Cuál modelo detecta mejor los productos en anaquel? Argumente por qué en el contexto de auditoría de stock se prefiere alta Precisión sobre alto Recall (o viceversa), y sustente su posición con los números obtenidos.

* **Recomendación final:** ¿Cuál modelo desplegaría en producción on-premise con CPU de tienda y por qué? Conecte explícitamente esta decisión con lo analizado en el Task 2.

* **Análisis de viabilidad operativa:** ¿Los FPS obtenidos son suficientes para el requerimiento de procesar una imagen cada 30 segundos? ¿Cuántas imágenes por hora podría procesar el sistema? ¿Cómo cambia esa viabilidad si la tienda instala 5 cámaras simultáneas?

* **¿Cuánto “cuesta” en MB cada punto de mAP?** Calcule la razón $MB/mAP$ para cada modelo y argumente si el modelo más pesado se justifica en el contexto de hardware limitado de VisorShelf.

* **Reflexión sobre generalización:** ¿Funcionaría el modelo igual en tiendas con diferente iluminación, cámaras de distinta resolución o categorías de productos distintas a las del dataset de entrenamiento? ¿Qué implicaciones tiene eso para la estrategia de expansión de VisorShelf?

* **Limitaciones del experimento:** Identifique al menos dos limitaciones metodológicas de su experimento que deberían resolverse antes de un despliegue real.